In [0]:
%run ./import_libraries

In [0]:
%run ./schema_registry

In [0]:
# Read control_load table to get list of tables to process
control_df = spark.table("adwm_wh.utilities.control_load")

# Filter for active tables only
active_tables = control_df.filter(col("isactive") == "Y").collect()

# print(f"Found {len(active_tables)} active tables to process:")
# for row in active_tables:
#     print(f"  - {row['schema']}.{row['table']}")

In [0]:
# Configuration
CATALOG = "adwm_wh"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
CHECKPOINT_BASE_PATH = "/Volumes/adwm_wh/volumes/checkpoints/silver"  # Checkpoint location for streaming state

print(f"Starting Silver Layer Processing...")
print(f"Catalog: {CATALOG}")
print(f"Source Schema: {BRONZE_SCHEMA}")
print(f"Target Schema: {SILVER_SCHEMA}")
print(f"Checkpoint Path: {CHECKPOINT_BASE_PATH}")
print(f"\n{'='*60}\n")

# Process each active table from bronze to silver
processed_tables = []
failed_tables = []
streaming_queries = []

for row in active_tables:
    database = row['database']
    schema_name = row['schema']
    table_name = row['table']
    table_key = f"{schema_name}.{table_name}".lower()
    
    # Get key columns for merge operation
    key_columns_str = row['keys']
    key_columns = [k.strip() for k in key_columns_str.split(',')]
    
    try:
        # Construct paths and table names
        bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        silver_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        checkpoint_path = f"{CHECKPOINT_BASE_PATH}/{schema_name}_{table_name}"
        
        print(f"Processing: {table_key}")
        print(f"  Source: {bronze_table}")
        print(f"  Target: {silver_table}")
        print(f"  Key columns: {', '.join(key_columns)}")
       
        # Read from bronze table in streaming mode
        bronze_df = spark.readStream.table(bronze_table)
        
        # Identify business columns (exclude bronze metadata)
        bronze_metadata_cols = ["source_file_name", "source_file_path", "source_file_timestamp", "ingestion_timestamp"]
        business_cols = [c for c in bronze_df.columns if c not in bronze_metadata_cols]
        
        print(f"  Business columns: {len(business_cols)}")
        
        # Remove duplicates and handle nulls - select only business columns
        silver_df = (
            bronze_df
            .select(*business_cols)  # Select only business columns (exclude bronze metadata)
            .dropDuplicates(business_cols)  # Remove exact duplicates
            .fillna("UNKNOWN")  # Replace all string nulls with 'UNKNOWN'
            .fillna(0)  # Replace all numeric nulls with 0
            .withColumn("silver_processing_timestamp", current_timestamp())
            .withColumn("data_quality_flag", lit("VALID"))
        )
        
        # Define merge function for foreachBatch
        def merge_to_silver(batch_df, batch_id):
            batch_count = batch_df.count()
            print(f"    Batch {batch_id}: Processing {batch_count} records")
            
            # Create merge condition based on key columns
            merge_condition = " AND ".join([f"target.{col} = source.{col}" for col in key_columns])
            
            # Use Delta merge
            from delta.tables import DeltaTable
            
            # Check if silver table exists, if not create it
            if not spark.catalog.tableExists(silver_table):
                print(f"    Creating new table: {silver_table}")
                batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
            else:
                # Get DeltaTable
                delta_table = DeltaTable.forName(spark, silver_table)
                
                # Perform merge
                print(f"    Merging records using condition: {merge_condition}")
                delta_table.alias("target").merge(
                    batch_df.alias("source"),
                    merge_condition
                ).whenMatchedUpdateAll(
                ).whenNotMatchedInsertAll(
                ).execute()
            
            print(f"    Batch {batch_id}: Completed")
        
        # Write to silver Delta table with streaming merge
        query = (
            silver_df.writeStream
            .format("delta")
            .foreachBatch(merge_to_silver)  # Use merge instead of append
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)  # Process all available data then stop
            .start()
        )
        
        print(f"  Streaming query started (ID: {query.id})")
        
        # Wait for this table's streaming query to complete
        query.awaitTermination()
        
        print(f"  ✓ Successfully processed {table_key}\n")
        processed_tables.append(table_key)
        streaming_queries.append((table_key, query.id))
        
    except Exception as e:
        print(f"  ✗ Error processing {table_key}: {str(e)}\n")
        failed_tables.append((table_key, str(e)))

print(f"\n{'='*60}")
print(f"Silver Layer Processing Summary:")
print(f"{'='*60}")
print(f"  Total tables attempted: {len(active_tables)}")
print(f"  Successfully processed: {len(processed_tables)} tables")
print(f"  Failed: {len(failed_tables)} tables")

if failed_tables:
    print(f"\n❌ Failed tables:")
    for table, error in failed_tables:
        print(f"  - {table}")
        print(f"    Error: {error[:150]}...")  # Truncate long errors

if processed_tables:
    print(f"\n✅ Successfully processed tables written to {CATALOG}.{SILVER_SCHEMA}:")
    for table in processed_tables:
        print(f"  - {table}")

print(f"\n{'='*60}")
print(f"Processing Complete!")
print(f"{'='*60}")

In [0]:
%sql
-- Verify data was loaded into silver tables
-- Check one table as example
-- SELECT * FROM adwm_wh.silver.Address LIMIT 10;

-- Check for any remaining null values
-- SELECT 
--   COUNT(*) as total_records,
--   COUNT(CASE WHEN column_name IS NULL THEN 1 END) as null_count_column_name
-- FROM adwm_wh.silver.Address;